### Train

In [22]:
# =========================
# Training Loop
# =========================
model = ImprovedTemporalAutoencoder(input_dim = X_tr_aug.shape[-1])
dataloader = train_loader


# ============================================================
# ============================================================
epochs=0
# ============================================================
# ============================================================

lr=1e-3 
device=DEV
patience=20

model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_val_loss = float('inf')
patience_counter = 0


#if ablation > 0  # Train the model just once, when the ablation is 0
for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in dataloader:
            #x = batch.to(device)  # (B, T, F)
            x, y = batch 
    
    # Move individual tensors to the device
            x = x.to(device)
            y = y.to(device)
    


            x_masked, mask = apply_mask(x, mask_ratio=0.2)

            emb, recon = model(x_masked)

            loss = masked_mse_loss(recon, x, mask)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)

        print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {val_loss:.4f} | patience_counter: {patience_counter}")

        # ---- Early Stopping ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            # save best model
            torch.save(model.state_dict(), "best_model1.pt")

        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# ============================================================
# ============================================================
# ============================================================
    # load best model
#model.load_state_dict(torch.load("best_model1.pt"))
# ============================================================
# ============================================================
# ============================================================
